# 🧬 UniProt Cancer Protein Explorer
### Programmatic Access to UniProt Data Notebook

---

**What this notebook does:**
- Connects to the UniProt REST API (Application Programming Interface) — a web service that lets us fetch protein data with code
- Searches for human cancer-related proteins (starting with the famous TP53 tumour suppressor gene)
- Displays the results in a clean, readable table
- Shows you how to check response headers (metadata about the results) and status codes (whether the request worked)

**How to use:**
1. Click **Runtime > Run All** from the menu above, OR
2. Run each cell one by one by clicking the ▶ button on the left

You can change the search term in **Step 3** to explore any cancer gene you like!

---

## ⚙️ Step 1: Install & Import Libraries
Libraries are pre-built tools that save us writing code from scratch.
- `requests` — lets Python talk to websites and APIs
- `pandas` — helps us organise data into tables
- `IPython.display` — makes our output look nice in the notebook

In [1]:
# Install pandas if not already available (Colab usually has it)
!pip install requests pandas --quiet

# Import the libraries we need
import requests  # For making API calls
import pandas as pd  # For displaying data in tables
from IPython.display import display, HTML  # For pretty output

print('✅ Libraries loaded successfully!')

✅ Libraries loaded successfully!


## 🔧 Step 2: Set Up the UniProt API Connection
The UniProt REST API (Representational State Transfer — Application Programming Interface) base URL is:
> `https://rest.uniprot.org`

We will specifically use the **UniProtKB (UniProt KnowledgeBase)** endpoint — this is the main database of reviewed and unreviewed protein entries.

In [2]:
# --- API Base URL ---
# This is the main address of the UniProt REST API
WEBSITE_API = 'https://rest.uniprot.org'

# UniProtKB (UniProt KnowledgeBase) endpoint — where all protein entries live
UNIPROTKB_API = f'{WEBSITE_API}/uniprotkb'

# --- Helper Function ---
# A function is a reusable block of code. This one handles our API requests.
def get_url(url, params=None):
    """
    Sends a GET request to the given URL and returns the response.
    Also checks the HTTP (HyperText Transfer Protocol) status code to
    make sure the request was successful.
    """
    response = requests.get(url, params=params)

    # HTTP Status Code check:
    # 2xx = Success ✅ | 4xx = Your error ❌ | 5xx = Server error ❌
    if response.status_code == 200:
        print(f'✅ Status Code: {response.status_code} — Request successful!')
    elif 400 <= response.status_code < 500:
        print(f'❌ Status Code: {response.status_code} — Error on your side. Check your query/URL (Uniform Resource Locator).')
        print(f'   Details: {response.text}')
        return None
    elif response.status_code >= 500:
        print(f'❌ Status Code: {response.status_code} — Error on UniProt server side. Try again later.')
        return None

    return response

print('✅ API setup complete!')

✅ API setup complete!


## 🔍 Step 3: Search for Cancer-Related Human Proteins

We will search for **TP53** — the most studied tumour suppressor gene in cancer research, often called the *'guardian of the genome'*.

The query format comes directly from UniProt's Advanced Search UI (User Interface):
> `gene:TP53 AND organism_id:9606 AND reviewed:true`

- `gene:TP53` — search by gene name
- `organism_id:9606` — taxonomy ID (identifier) for *Homo sapiens* (Human)
- `reviewed:true` — only Swiss-Prot reviewed (high quality, manually annotated) entries

💡 **To customise:** Change `SEARCH_GENE` below to any cancer gene e.g. `BRCA1`, `EGFR`, `MYC`, `KRAS`

In [3]:
# ✏️ CHANGE THIS to explore different cancer genes!
SEARCH_GENE = 'TP53'

# Build the search query (this is the SOLR/Lucene format the webinar mentioned)
query = f'gene:{SEARCH_GENE} AND organism_id:9606 AND reviewed:true'

# Define what fields (columns) we want back from the API
fields = 'accession,gene_names,protein_name,organism_name,length,annotation_score'

# Set up the request parameters
params = {
    'query': query,
    'fields': fields,
    'format': 'json',  # Ask for JSON (JavaScript Object Notation) format — easy to read in Python
    'size': 10         # Return up to 10 results
}

print(f'🔍 Searching UniProtKB for: "{query}"\n')

# Make the API call
response = get_url(UNIPROTKB_API + '/search', params=params)

🔍 Searching UniProtKB for: "gene:TP53 AND organism_id:9606 AND reviewed:true"

✅ Status Code: 200 — Request successful!


## 📋 Step 4: Read the Response Headers (Metadata)

As the webinar covered, **response headers** act as metadata — they tell us important information *about* our results:
- `x-total-results` — the **total** number of matching proteins in UniProt (not just the ones we downloaded)
- `x-uniprot-release` — which version of UniProt this data comes from
- `content-type` — the format of the data returned

In [4]:
if response:
    headers = response.headers

    print('📋 RESPONSE HEADERS (Metadata about our results):')
    print('=' * 55)

    # x-total-results: total number of matching entries in UniProt
    total = headers.get('x-total-results', 'N/A')
    print(f'🔢 x-total-results    : {total} matching proteins found in UniProt')

    # x-uniprot-release: the database release version
    release = headers.get('x-uniprot-release', 'N/A')
    print(f'📦 x-uniprot-release  : {release}')

    # content-type: the format of data returned (should be JSON)
    content = headers.get('content-type', 'N/A')
    print(f'📄 content-type       : {content}')

    print('=' * 55)
    print(f'\n💡 Note: We requested 10 results, but there are {total} total matches.')
    print('   In a full pipeline, you would paginate (page through) all results using the "link" header.')

📋 RESPONSE HEADERS (Metadata about our results):
🔢 x-total-results    : 1 matching proteins found in UniProt
📦 x-uniprot-release  : 2026_01
📄 content-type       : application/json

💡 Note: We requested 10 results, but there are 1 total matches.
   In a full pipeline, you would paginate (page through) all results using the "link" header.


## 📊 Step 5: Display the Protein Results as a Table

Now let's parse (read and extract) the JSON (JavaScript Object Notation) response and display it as a clean table!

In [5]:
if response:
    # Parse the JSON response into a Python dictionary
    data = response.json()
    results = data.get('results', [])

    if not results:
        print('⚠️ No results found. Try a different gene name in Step 3.')
    else:
        # Build a list of rows for our table
        rows = []
        for entry in results:
            # Extract accession (UniProt unique ID)
            accession = entry.get('primaryAccession', 'N/A')

            # Extract gene name
            genes = entry.get('genes', [])
            gene_name = genes[0].get('geneName', {}).get('value', 'N/A') if genes else 'N/A'

            # Extract protein name
            prot_desc = entry.get('proteinDescription', {})
            recommended = prot_desc.get('recommendedName', {})
            protein_name = recommended.get('fullName', {}).get('value', 'N/A') if recommended else 'N/A'

            # Extract organism
            organism = entry.get('organism', {}).get('scientificName', 'N/A')

            # Extract protein length (number of amino acids)
            length = entry.get('sequence', {}).get('length', 'N/A')

            # Extract annotation score (quality indicator, out of 5)
            ann_score = entry.get('annotationScore', 'N/A')

            rows.append({
                'Accession (UniProt ID)': accession,
                'Gene': gene_name,
                'Protein Name': protein_name,
                'Organism': organism,
                'Length (aa)': length,
                'Annotation Score (/5)': ann_score
            })

        # Convert to a pandas DataFrame (a table structure)
        df = pd.DataFrame(rows)

        print(f'\n🧬 Cancer Protein Results for Gene: {SEARCH_GENE} | Organism: Homo sapiens (Human)')
        print(f'   Showing {len(df)} of {total} total results\n')
        display(df)


🧬 Cancer Protein Results for Gene: TP53 | Organism: Homo sapiens (Human)
   Showing 1 of 1 total results



,Accession (UniProt ID),Gene,Protein Name,Organism,Length (aa),Annotation Score (/5)
0,P04637,TP53,Cellular tumor antigen p53,Homo sapiens,393,5.0


## 🔗 Step 6: Generate UniProt Links for Each Protein

Each accession (UniProt ID) links directly to a full protein entry page. Let's make those links clickable!

In [6]:
if response and results:
    print('🔗 Direct UniProt links for each protein found:\n')
    for _, row in df.iterrows():
        acc = row['Accession (UniProt ID)']
        gene = row['Gene']
        protein = row['Protein Name']
        link = f'https://www.uniprot.org/uniprotkb/{acc}'
        print(f'  🧬 {gene} | {protein}')
        print(f'     🔗 {link}\n')

🔗 Direct UniProt links for each protein found:

  🧬 TP53 | Cellular tumor antigen p53
     🔗 https://www.uniprot.org/uniprotkb/P04637



## 💾 Step 7: Save Results to a CSV File

CSV (Comma-Separated Values) is a simple spreadsheet format you can open in Excel or Google Sheets.

In [7]:
if response and results:
    filename = f'UniProt_{SEARCH_GENE}_human_proteins.csv'
    df.to_csv(filename, index=False)
    print(f'✅ Results saved to: {filename}')
    print('   You can download this file from the Files panel on the left sidebar in Colab.')

✅ Results saved to: UniProt_TP53_human_proteins.csv
   You can download this file from the Files panel on the left sidebar in Colab.


---
## 🎓 What You Just Did — Summary

Congratulations! You just used the **UniProt REST API** programmatically. Here's what each step demonstrated:

| Step | What it showed |
|------|----------------|
| Step 1 | Importing Python libraries (tools) |
| Step 2 | Setting up the API (Application Programming Interface) base URL and a helper function |
| Step 3 | Building a SOLR query and sending a GET request |
| Step 4 | Reading response headers as metadata (x-total-results, release version) |
| Step 5 | Parsing JSON (JavaScript Object Notation) results into a table |
| Step 6 | Generating direct UniProt links for deeper exploration |
| Step 7 | Saving results to a CSV (Comma-Separated Values) file |

---
### 🚀 Next Steps — Try These!
- Change `SEARCH_GENE` in Step 3 to `BRCA1`, `KRAS`, `EGFR`, or `MYC`
- Increase `size` from `10` to `25` to get more results
- Remove `AND reviewed:true` to include unreviewed entries too
- Explore the full API documentation at: https://www.uniprot.org/help/api

---
*Built using the UniProt REST API — www.uniprot.org | EMBL-EBI (European Molecular Biology Laboratory – European Bioinformatics Institute)*